# 08 · PGFN — Lista de Empresas com Dívida Previdenciária ou FGTS

**Objetivo:** a partir dos Dados Abertos da Dívida Ativa (PGFN) baixados em
`data/raw/pgfn/` (CLI `python -m src.pgfn`), consolidar **uma linha por
empresa (CNPJ)** que apareceu como devedora **previdenciária** ou de **FGTS**
em qualquer trimestre de 2020 até o mais recente.

**Saída:** `outputs/tables/pgfn_empresas_devedoras.{csv,parquet}`.

**Como funciona:** `src.pgfn.agregar_empresas` lê os CSVs **em streaming de
dentro dos .zip** (não extrai pro disco), filtra `TIPO_PESSOA == 'Pessoa
jurídica'` e agrega por `CPF_CNPJ`.

**Limitações:** reflete o *estoque* publicado em cada trimestre (se a empresa
quitou, some nos trimestres seguintes — por isso olhamos o período todo).

In [ ]:
# Preâmbulo: torna o pacote src importável a partir do notebook
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.config import load_config
from src import pgfn
cfg = load_config()
destino = pathlib.Path(cfg.get('pgfn', {}).get('dir', 'data/raw/pgfn'))
if not destino.is_absolute():
    destino = cfg['root'] / destino
print('Lendo de:', destino)

In [ ]:
# Confere o que foi baixado (um .zip por trimestre x tipo)
zips = sorted(destino.rglob('Dados_abertos_*.zip'))
print(len(zips), 'arquivos .zip encontrados')
for z in zips[:4] + zips[-2:]:
    print(' ', z.relative_to(destino), f'{z.stat().st_size/1e6:.0f} MB')

In [ ]:
# Agrega TODAS as empresas (PJ) com dívida previdenciária ou FGTS no período.
# Streaming; leva alguns minutos. Teste rápido: tipos=['FGTS'].
linhas = pgfn.agregar_empresas(destino, tipos=['Previdenciario', 'FGTS'], apenas_pj=True)
df = pd.DataFrame(linhas)
print(f'\n{len(df):,} empresas únicas com alguma dívida previdenciária ou FGTS')
df.head(10)

In [ ]:
# Visão geral + recorte de dívida EXIGÍVEL (em cobrança) vs suspensa/parcelada
print('Com dívida PREVIDENCIÁRIA :', (df.TEVE_PREVIDENCIARIO == 'S').sum())
print('Com dívida FGTS          :', (df.TEVE_FGTS == 'S').sum())
print('Com AMBAS                :', ((df.TEVE_PREVIDENCIARIO=='S') & (df.TEVE_FGTS=='S')).sum())
print('Com dívida EXIGÍVEL      :', (df.DIVIDA_EXIGIVEL == 'S').sum(), '(em cobrança no últ. trim.)')
print()
print('Soma por situação (R$ bi, snapshot do últ. trim. de cada empresa):')
for c in ['VALOR_EM_COBRANCA','VALOR_PARCELADO_BENEF','VALOR_GARANTIDO','VALOR_SUSPENSO_JUD']:
    print(f'  {c:<22} {df[c].sum()/1e9:>10,.2f}')

In [ ]:
# Exemplo: empresas grandes que NÃO têm dívida exigível (tudo parcelado/garantido)
cols = ['CPF_CNPJ','NOME_DEVEDOR','UF','VALOR_EM_COBRANCA','VALOR_PARCELADO_BENEF','VALOR_GARANTIDO','DIVIDA_EXIGIVEL']
df[df.DIVIDA_EXIGIVEL=='N'].sort_values('VALOR_TOTAL_REF', ascending=False)[cols].head(10)

In [ ]:
# Salva a lista consolidada
out = cfg['abs']['tables']
csv_path = out / 'pgfn_empresas_devedoras.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
try:
    df.to_parquet(out / 'pgfn_empresas_devedoras.parquet', index=False)
except Exception as e:
    print('(parquet opcional falhou:', e, ')')
print('Salvo em:', csv_path)